# 02 — Room occupancy under a sensor outage
**Assessed group practical • three students • English**

Use the worked sprinkler notebook as a recipe. Complete the **four short modelling
cells** below; supplied helper functions handle data loading and scoring.
Do not write a new inference engine, data scraper or preprocessing pipeline.

The task is to compare the same fitted model on the same test records with full
sensor evidence and with one assigned sensor feature omitted. Explain the result,
including a small or unexpected effect. There is no performance leaderboard.

**Submission:** this notebook, saved with outputs, and three presentation slides in
PDF. Common deadline: end of lecture 2. No separate report or
individual essay. The five-minute oral will involve questions for each member of the group.

This is a starter: cells tagged `student-task` deliberately raise
`NotImplementedError` until completed. Real prepared data are supplied by the
lecturer before class. The notebook does not download data or use a synthetic
fallback for the assessment.

In [ ]:
!pip install pandas pgmpy

In [2]:
# Utility functions, run this cell first. Do not edit it. The code is provided for your convenience and to ensure consistency across student submissions.

"""Supplied infrastructure for the occupancy practical; modelling stays in the notebook."""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from collections.abc import Callable, Mapping
import hashlib
import json
import numpy as np
import pandas as pd

FEATURES = ['T', 'L', 'S', 'C', 'M']
VARIABLES = ['O', *FEATURES]
THRESHOLD = 1.0 / 6.0


@dataclass(frozen=True)
class CourseData:
    train: pd.DataFrame
    test: pd.DataFrame
    state_names: dict[str, list[int]]
    manifest: dict


def locate_data(start: Path | None = None) -> Path:
    """Find the distributed data folder from a notebook or the package root."""
    here = (start or Path.cwd()).resolve()
    for folder in [here / 'data', here.parent / 'data', here]:
        if (folder / 'manifest.json').is_file():
            return folder
    raise FileNotFoundError(
        'Prepared occupancy data are missing. Ask the lecturer for the data folder. '
        'Before class, the lecturer runs: python tools/prepare_occupancy.py '
        '--input Occupancy_Estimation.csv --output-dir data. '
        'The sprinkler example does not require these data.'
    )


def load_course_data(folder: Path | str, *, allow_synthetic: bool = False) -> CourseData:
    folder = Path(folder)
    meta = json.loads((folder / 'manifest.json').read_text(encoding='utf-8'))
    if meta['source_kind'] != 'uci864' and not allow_synthetic:
        raise ValueError(
            'This is a synthetic test fixture, not the assessed UCI dataset.')
    frames = []
    for split in ['train', 'test']:
        p = folder / f'occupancy_{split}.csv'
        actual = hashlib.sha256(p.read_bytes()).hexdigest()
        if actual != meta['files'][p.name]['sha256']:
            raise ValueError(f'{p.name} differs from the published manifest.')
        frame = pd.read_csv(p)
        required = ['record_id', 'timestamp', *VARIABLES]
        if list(frame.columns) != required:
            raise ValueError(
                f'Unexpected columns in {p.name}. Expected {required}.')
        if frame[required].isna().any().any():
            raise ValueError(f'Missing training/test values in {p.name}.')
        for var in VARIABLES:
            legal = meta['state_names'][var]
            if not frame[var].isin(legal).all():
                raise ValueError(f'Unknown state of {var} in {p.name}.')
            frame[var] = frame[var].astype(int)
        if len(frame) != meta['splits'][split]['rows']:
            raise ValueError(f'Wrong row count in {p.name}.')
        if not pd.to_datetime(frame.timestamp).is_monotonic_increasing:
            raise ValueError(f'{p.name} is not chronologically sorted.')
        frames.append(frame)
    train, test = frames
    if set(train.record_id) & set(test.record_id):
        raise ValueError('Training and test records overlap.')
    if pd.to_datetime(train.timestamp).max() >= pd.to_datetime(test.timestamp).min():
        raise ValueError(
            'The test period must follow the training period strictly.')
    return CourseData(train, test, meta['state_names'], meta)


def outage_for_group(group: str) -> str:
    """Assignment fixed before any student result is seen."""
    group = group.strip().upper()
    if not (len(group) == 3 and group[0] == 'G' and group[1:].isdigit()):
        raise ValueError('Use a group identifier from G01 to G19.')
    number = int(group[1:])
    if not 1 <= number <= 19:
        raise ValueError('Use a group identifier from G01 to G19.')
    return 'C' if number <= 4 else 'M' if number <= 8 else 'L' if number <= 12 else 'S' if number <= 16 else 'T'


def predict_cached(rows: pd.DataFrame,
                   make_evidence: Callable[[pd.Series, str | None], Mapping[str, int]],
                   probability: Callable[[Mapping[str, int]], float],
                   omitted: str | None = None) -> np.ndarray:
    """Call the student's query once per unique evidence pattern, preserving row order.

    The helper checks that the evidence is exactly the declared sensor subset;
    it never reads O as evidence and never learns model parameters.
    """
    if omitted is not None and omitted not in FEATURES:
        raise ValueError('The omitted variable must be a sensor feature.')
    expected_keys = set(FEATURES) - ({omitted} if omitted else set())
    cache: dict[tuple, float] = {}
    result = []
    # Give student code only sensor columns: targets and metadata are unavailable here.
    for _, row in rows[FEATURES].iterrows():
        evidence = dict(make_evidence(row, omitted))
        if set(evidence) != expected_keys:
            raise ValueError(
                f'Evidence keys must be {sorted(expected_keys)}; got {sorted(evidence)}.')
        key = tuple(sorted((k, int(v)) for k, v in evidence.items()))
        if key not in cache:
            p = float(probability(dict(key)))
            if not np.isfinite(p) or not 0 <= p <= 1:
                raise ValueError(
                    'An occupancy posterior is nonfinite or outside [0,1].')
            cache[key] = p
        result.append(cache[key])
    return np.asarray(result, dtype=float)


def binary_metrics(y: np.ndarray | pd.Series, p: np.ndarray,
                   threshold: float = THRESHOLD) -> dict[str, float | int]:
    y = np.asarray(y)
    p = np.asarray(p, dtype=float)
    if y.ndim != 1 or p.shape != y.shape or len(y) == 0:
        raise ValueError(
            'Labels and probabilities must be nonempty aligned vectors.')
    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError(
            'Use binary labels and finite probabilities in [0,1].')
    if not 0 <= threshold <= 1:
        raise ValueError('Threshold must be in [0,1].')
    decisions = p >= threshold  # Occupied is the declared tie decision.
    tn = int(np.sum((y == 0) & ~decisions))
    fp = int(np.sum((y == 0) & decisions))
    fn = int(np.sum((y == 1) & ~decisions))
    tp = int(np.sum((y == 1) & decisions))
    return {'n': len(y), 'brier': float(np.mean((p-y)**2)),
            'mean_loss': float((fp+5*fn)/len(y)),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}


def results_table(y: pd.Series, predictions: Mapping[str, np.ndarray]) -> pd.DataFrame:
    return pd.DataFrame({name: binary_metrics(y, p) for name, p in predictions.items()}).T.rename_axis('condition')

In [3]:
from pathlib import Path
import sys
from importlib.metadata import version
import numpy as np
import pandas as pd
from IPython.display import display
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination


GROUP = "G01"  # Replace with your assigned identifier.
MEMBERS = ["Name 1", "Name 2", "Name 3"]
OMITTED = outage_for_group(GROUP)
print("Group:", GROUP, "outage:", OMITTED, "pgmpy:", version("pgmpy"))

Group: G01 outage: C pgmpy: 1.1.2


/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


## 1. Read the supplied data contract
`O=1` means at least one occupant. Sensor features are temperature `T`, light `L`,
sound `S`, CO₂ `C` and motion `M`. Continuous features were discretised using **only
the earlier training period**; use the recorded states, not new test-fitted bins.

`record_id` and `timestamp` identify the rows; neither is evidence.
The original occupancy count and the derived label `O` are never sensor evidence.

In [5]:
data = load_course_data(locate_data())
train, test, state_names = data.train, data.test, data.state_names
print("Protocol:", data.manifest["protocol"])
print("Source hash:", data.manifest["source_sha256"])
print("Split:", data.manifest["splits"])
print("State dictionary:", state_names)
display(train.head())

Protocol: imt-bn-two-mornings-v1
Source hash: c090ee5c94b61762bfec9d22767864490b51030f98cca030f02822468db25d9c
Split: {'train': {'rows': 7090, 'first_timestamp': '2017-12-22T10:49:41', 'last_timestamp': '2017-12-26T00:35:20', 'class_counts': {'0': 5483, '1': 1607}}, 'test': {'rows': 3039, 'first_timestamp': '2017-12-26T00:35:51', 'last_timestamp': '2018-01-11T09:00:09', 'class_counts': {'0': 2745, '1': 294}}}
State dictionary: {'O': [0, 1], 'M': [0, 1], 'T': [0, 1, 2], 'L': [0, 1], 'S': [0, 1, 2], 'C': [0, 1, 2]}


,record_id,timestamp,O,T,L,S,C,M
0,0,2017-12-22T10:49:41,1,0,1,2,1,0
1,1,2017-12-22T10:50:12,1,0,1,2,1,0
2,2,2017-12-22T10:50:42,1,0,1,2,1,0
3,3,2017-12-22T10:51:13,1,0,1,2,1,0
4,4,2017-12-22T10:51:44,1,0,1,2,1,0


## 2. Task A — Represent the fixed model
Use the factorisation
\[
P(O,T,L,S,C,M)=P(O)P(T\mid O)P(L\mid O)P(S\mid O)P(C\mid O)P(M\mid O).
\]
Create the five edges and the model. Use the same constructor as in sprinkler section 2.
This assumes conditional independence of sensor features given occupancy.

In [ ]:
# Task A: define edges and model; retain all six VARIABLES.
# Example API pattern: DiscreteBayesianNetwork(list_of_parent_child_pairs).
raise NotImplementedError("Create edges and model by transposing sprinkler section 2.")

In [ ]:
assert set(model.nodes()) == set(VARIABLES)
assert set(model.edges()) == {("O", feature) for feature in FEATURES}

## 3. Task B — Learn CPDs, rather than copying sprinkler probabilities
Transpose sprinkler section 8. Construct a `BayesianEstimator` from `model`,
`train[VARIABLES]` and `state_names`. Request Dirichlet posterior-mean estimates
with **one pseudocount per cell**, then attach the returned CPDs.
Use `n_jobs=1` for this small model.

In [ ]:
# Task B: estimate CPDs using only train[VARIABLES] and attach them to model.
raise NotImplementedError("Adapt the BayesianEstimator recipe from sprinkler section 8.")

In [ ]:
assert model.check_model()
for cpd in model.get_cpds():
    np.testing.assert_allclose(cpd.get_values().sum(axis=0), 1.0, atol=1e-12, rtol=0)

# Supplied numerical check; explain the numerator and denominator orally.
child, child_state, parent_state = "M", 1, 1
subset = train.loc[train.O == parent_state, child]
manual = (int((subset == child_state).sum()) + 1) / (len(subset) + len(state_names[child]))
computed = float(model.get_cpds(child).get_value(**{child: child_state, "O": parent_state}))
np.testing.assert_allclose(manual, computed, atol=1e-12, rtol=0)
print("P(M=1 | O=1), hand count and CPD:", manual, computed)

## 4. Task C — Query occupancy
Create a `VariableElimination` object. Complete the function that returns
$P(O=1\mid\mathrm{evidence})$. Adapt the query and named-state access in sprinkler
sections 5–6. Evidence is a dictionary of observed sensor states.

In [ ]:
inference = VariableElimination(model)

def occupancy_probability(evidence: dict[str, int]) -> float:
    """Return the state-1 posterior; O itself must not be in evidence."""
    if not set(evidence).issubset(FEATURES):
        raise ValueError("Evidence may contain sensor features only.")
    # Query O and return float(result.get_value(O=1)).
    raise NotImplementedError("Transpose the posterior query from the sprinkler example.")

In [ ]:
# The same smoothed root CPD defines the prior-only baseline and no-evidence check.
prior = float(model.get_cpds("O").get_value(O=1))
np.testing.assert_allclose(occupancy_probability({}), prior, atol=1e-12, rtol=0)
print("Fitted occupancy prior:", prior)

## 5. Task D — Form evidence; omit an unavailable sensor
Return all five sensor states for the full model. When `omitted` is a sensor name,
leave its key out. A missing motion sensor is **not** `M=0`.
The supplied predictor passes only sensor columns to this function and checks its keys.

In [ ]:
def make_evidence(row: pd.Series, omitted: str | None = None) -> dict[str, int]:
    """Read FEATURES from this row, except the variable named in omitted."""
    raise NotImplementedError("Build the dictionary; omit the unavailable key rather than zero-filling.")

In [ ]:
# All three conditions use exactly these test records, in this order.
predictions = {
    "prior only": np.full(len(test), prior),
    "full sensors": predict_cached(test, make_evidence, occupancy_probability),
    f"without {OMITTED}": predict_cached(test, make_evidence, occupancy_probability, omitted=OMITTED),
}
summary = results_table(test.O, predictions)
display(summary.round(5))

## 6. Interpret the experiment
The helper reports Brier score (mean squared probability error), confusion counts
and mean loss. The supplied hypothetical costs are false occupied = 1, false empty = 5,
correct decisions = 0. The fixed decision is occupied for $p\ge1/6$ (including ties).
These are educational costs, not measured building savings.

**Complete these short group notes:**

- Which comparison isolates the effect of omitting your assigned sensor?
- What changed, by how much, and what remained controlled?
- What does the result suggest, and what important claim does it not establish?

Do not tune the graph, bins, smoothing or threshold on the test outcomes.

In [ ]:
# A reproducible incident per group, assigned without inspecting model scores.
group_index = int(GROUP[1:]) - 1
incident_position = min(len(test)-1, int((group_index + 0.5) * len(test) / 19))
incident = test.iloc[incident_position]
evidence_full = make_evidence(incident[FEATURES], None)
evidence_outage = make_evidence(incident[FEATURES], OMITTED)
p_full = occupancy_probability(evidence_full)
p_outage = occupancy_probability(evidence_outage)
display(pd.DataFrame({
    "condition": ["full sensors", f"without {OMITTED}"],
    "probability": [p_full, p_outage],
    "decision_occupied": [p_full >= THRESHOLD, p_outage >= THRESHOLD],
}))
print("Incident record:", int(incident.record_id), "timestamp:", incident.timestamp)
print("Evidence:", evidence_full, "versus", evidence_outage)

### Three-slide presentation (Optional)
1. **Model and assumption:** graph, variable meanings and one modelling assumption.
2. **Protocol and evidence:** the fixed split, readable prior/full/outage results table,
   and one correctly interpreted numerical result.
3. **Engineering conclusion:** the incident or outage consequence, an important
   limitation and one further validation.

### Data attribution
Adarsh Pal Singh and Sachin Chaudhari, **Room Occupancy Estimation**, UCI Machine
Learning Repository, DOI **10.24432/C5P605**, CC BY 4.0. The binary target, same-row
sensor aggregates, training-only discretisation and fixed split are course adaptations.